In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

# Зчитуємо orders дані
orders = pd.read_csv(root / "data" / "raw" / "orders.csv")
customers = pd.read_csv(root / "data" / "raw" / "customers.csv")
merchants = pd.read_csv(root / "data" / "raw" / "merchants.csv")

print("\nБАЗОВИЙ ОГЛЯД СТРУКТУРИ Orders\n")
display(orders.head())
display(orders.tail())
display(orders.sample(3))
print(f"Розмірність: {orders.shape}")
print(f"Колонки: {list(orders.columns)}")
print(f"Індекси: {orders.index}")
print("\nТипи даних:"); display(orders.dtypes)
print("\nІнформація:"); orders.info()

print("\nОПИСОВА СТАТИСТИКА ТА КАТЕГОРІЇ\n")
display(orders.describe(include='all'))
print("\nКількість унікальних значень у кожній колонці:"); display(orders.nunique())
print("\nРозподіл статусів замовлень:"); display(orders['order_status'].value_counts(dropna=False))
print("\nРозподіл валют:"); display(orders['currency'].value_counts(dropna=False))

import pandas.api.types as ptypes

print("\nПРОПУСКИ, ПОРОЖНІ РЯДКИ ТА АНОМАЛЬНІ ПРОБІЛИ\n")
missing_stats_orders = []

for col in orders.columns:
    null_cnt = orders[col].isna().sum()
    is_text = ptypes.is_string_dtype(orders[col]) or orders[col].dtype == 'object'
    
    if is_text:
        blank_cnt = orders[col].dropna().astype(str).str.strip().eq('').sum() # Рядки, що складаються з пробілів або абсолютно порожні
        space_inside_cnt = orders[col].astype(str).str.contains(' ', na=False).sum() # Для orders шукаємо будь-який пробіл всередині рядка
    else:
        blank_cnt = 0
        space_inside_cnt = 0
        
    missing_stats_orders.append({
        'Колонка': col, 
        'Пропуски': null_cnt, 
        'Частка NaN': round((null_cnt / len(orders)) * 100, 2),
        'Лише пробіли ("   ")': blank_cnt, 
        'Містять пробіл': space_inside_cnt
    })

display(pd.DataFrame(missing_stats_orders))

print("\nДеталізація пропусків ")
has_nans = False
for col in orders.columns:
    missing_mask = orders[col].isna()
    if missing_mask.sum() > 0:
        has_nans = True
        print(f"\nКолонка '{col}' має {missing_mask.sum()} пропущених значень. Ось ці рядки:")
        display(orders[missing_mask])

if not has_nans:
    print("Пропусків у цій таблиці не виявлено.")

print("\nЗнайдені аномалії з пробілами")
for col in orders.columns:
    if ptypes.is_string_dtype(orders[col]) or orders[col].dtype == 'object':
        
        mask_spaces = orders[col].astype(str).str.contains(' ', na=False)
        if mask_spaces.sum() > 0:
            print(f"\nКолонка '{col}' має {mask_spaces.sum()} значень із пробілами:")
            display(orders[mask_spaces][['order_id', col]].head())

print("\nДУБЛІКАТИ (РЯДКИ ТА КЛЮЧІ)\n")
full_dups = orders[orders.duplicated(keep=False)]
print(f"Повні дублікати (всі стовпці збігаються): {len(full_dups)}")
if not full_dups.empty: display(full_dups.sort_values(by=list(orders.columns)))

key_dups = orders[orders.duplicated(subset=['order_id'], keep=False)]
print(f"Дублікати кандидатного ключа 'order_id': {len(key_dups)}")
if not key_dups.empty: display(key_dups.sort_values(by='order_id'))

print("\nЧАСОВИЙ ДІАПАЗОН (created_at)\n")
parsed_dates = pd.to_datetime(orders['created_at'], errors='coerce')
invalid_dates = orders['created_at'].notna() & parsed_dates.isna()
print(f"Мінімальна дата: {parsed_dates.min()}")
print(f"Максимальна дата: {parsed_dates.max()}")
print(f"Пропущено дат (NaN): {orders['created_at'].isna().sum()}")
print(f"Некоректні дати (не парсяться): {invalid_dates.sum()}")
if invalid_dates.sum() > 0: 
    print("Приклади некоректних дат:"); display(orders[invalid_dates][['order_id', 'created_at']])

print("\nЕКСТРЕМАЛЬНІ ЗНАЧЕННЯ (MIN, MAX, 0, НЕГАТИВНІ)\n")
num_cols = ['subtotal_amount', 'delivery_fee', 'discount_amount', 'total_amount']
for col in num_cols:
    neg = (orders[col] < 0).sum()
    zeros = (orders[col] == 0).sum()
    print(f"[{col:<15}] Min: {orders[col].min():>8.2f} | Max: {orders[col].max():>9.2f} | Нулів: {zeros:>4} | Від'ємних: {neg:>2}")

print("\nПЕРЕВІРКА ЗВ'ЯЗКІВ (FOREIGN KEYS)\n")
# Перевіряємо чи є в orders клієнти, яких немає в customers
missing_customers = orders[~orders['customer_id'].isin(customers['customer_id'])]
print(f"Замовлення з відсутніми (невідомими) customer_id: {len(missing_customers)}")
if not missing_customers.empty: display(missing_customers[['order_id', 'customer_id']].head())

# Перевіряємо чи є в orders заклади, яких немає в merchants
missing_merchants = orders[~orders['merchant_id'].isin(merchants['merchant_id'])]
print(f"Замовлення з відсутніми (невідомими) merchant_id: {len(missing_merchants)}")
if not missing_merchants.empty: display(missing_merchants[['order_id', 'merchant_id']].head())


БАЗОВИЙ ОГЛЯД СТРУКТУРИ Orders



,order_id,customer_id,merchant_id,created_at,order_status,currency,subtotal_amount,delivery_fee,discount_amount,total_amount
0,S06-O0000001,S06-C000793,S06-M00013,2025-09-01T08:23:42Z,delivered,UAH,105.81,76.05,5.29,176.57
1,S06-O0000002,S06-C000377,S06-M00034,2025-09-01T09:14:26Z,delivered,UAH,1698.69,79.81,0.00,1778.50
2,S06-O0000003,S06-C000178,S06-M00016,2025-09-01T12:08:22Z,delivered,UAH,1049.04,0.00,0.00,1049.04
3,S06-O0000004,S06-C000529,S06-M00024,2025-09-01T12:44:22Z,delivered,UAH,46.30,68.02,0.00,114.32
4,S06-O0000005,S06-C000547,S06-M00004,2025-09-01T13:25:19Z,delivered,UAH,24.22,57.51,0.00,81.73


,order_id,customer_id,merchant_id,created_at,order_status,currency,subtotal_amount,delivery_fee,discount_amount,total_amount
5996,S06-O0005997,S06-C000736,S06-M00006,2026-08-31T19:44:07Z,delivered,UAH,690.07,83.90,103.51,670.46
5997,S06-O0005998,S06-C000667,S06-M00034,2026-08-31T19:46:33Z,delivered,UAH,728.36,69.86,0.00,798.22
5998,S06-O0005999,S06-C000774,S06-M00024,2026-08-31T21:08:24Z,delivered,UAH,408.09,70.62,61.21,417.50
5999,S06-O0006000,S06-C000579,S06-M00027,2026-08-31T22:18:30Z,delivered,UAH,802.16,82.05,0.00,884.21
6000,S06-O0003598,S06-C000211,S06-M00035,2026-04-04T13:42:00Z,delivered,UAH,941.40,0.00,188.28,753.12


,order_id,customer_id,merchant_id,created_at,order_status,currency,subtotal_amount,delivery_fee,discount_amount,total_amount
3068,S06-O0003069,S06-C000711,S06-M00015,2026-02-28T19:14:59Z,delivered,UAH,410.98,89.53,0.00,500.51
2764,S06-O0002765,S06-C000481,S06-M00031,2026-02-08T11:43:44Z,delivered,UAH,1364.25,61.53,272.85,1152.93
4789,S06-O0004790,S06-C000198,S06-M00009,2026-06-15T19:04:23Z,delivered,UAH,128.67,66.91,0.00,195.58


Розмірність: (6001, 10)
Колонки: ['order_id', 'customer_id', 'merchant_id', 'created_at', 'order_status', 'currency', 'subtotal_amount', 'delivery_fee', 'discount_amount', 'total_amount']
Індекси: RangeIndex(start=0, stop=6001, step=1)

Типи даних:


order_id               str
customer_id            str
merchant_id            str
created_at             str
order_status           str
currency               str
subtotal_amount    float64
delivery_fee       float64
discount_amount    float64
total_amount       float64
dtype: object

... (20 outputs left out)


ЕКСТРЕМАЛЬНІ ЗНАЧЕННЯ (MIN, MAX, 0, НЕГАТИВНІ)

[subtotal_amount] Min:     0.90 | Max:   3401.88 | Нулів:    0 | Від'ємних:  0
[delivery_fee   ] Min:     0.00 | Max:    135.82 | Нулів:  616 | Від'ємних:  0
[discount_amount] Min:     0.00 | Max:    350.00 | Нулів: 4327 | Від'ємних:  0
[total_amount   ] Min:     2.68 | Max:   3401.88 | Нулів:    0 | Від'ємних:  0

ПЕРЕВІРКА ЗВ'ЯЗКІВ (FOREIGN KEYS)

Замовлення з відсутніми (невідомими) customer_id: 1


,order_id,customer_id
3980,S06-O0003981,S06-C999999


Замовлення з відсутніми (невідомими) merchant_id: 1


,order_id,merchant_id
1793,S06-O0001794,S06-M99998


In [2]:
import pandas.api.types as ptypes

orders = pd.read_csv(root / "data" / "raw" / "orders.csv")
customers = pd.read_csv(root / "data" / "raw" / "customers.csv")
merchants = pd.read_csv(root / "data" / "raw" / "merchants.csv")

print("\nБАЗОВИЙ ОГЛЯД СТРУКТУРИ (customers.csv)\n")
display(customers.head())
display(customers.tail())
display(customers.sample(3))
print(f"Розмірність (shape): {customers.shape}")
print(f"Колонки (columns): {list(customers.columns)}")
print("\nТипи даних (dtypes):"); display(customers.dtypes)
print("\nІнформація (info):"); customers.info()

print("\nОПИСОВА СТАТИСТИКА ТА КАТЕГОРІЇ\n")
display(customers.describe(include='all'))
print("\nКількість унікальних значень у кожній колонці:"); display(customers.nunique())
print("\nРозподіл за містами (city):"); display(customers['city'].value_counts(dropna=False))
print("\nРозподіл за статусом (customer_status):"); display(customers['customer_status'].value_counts(dropna=False))

print("\nПРОПУСКИ, ПОРОЖНІ РЯДКИ ТА АНОМАЛЬНІ ПРОБІЛИ\n")
missing_stats_cust = []

for col in customers.columns:
    null_cnt = customers[col].isna().sum()
    is_text = ptypes.is_string_dtype(customers[col]) or customers[col].dtype == 'object'
    
    if is_text:
        blank_cnt = customers[col].dropna().astype(str).str.strip().eq('').sum() # Рядки, що складаються з пробілів або абсолютно порожні
        edge_spaces_cnt = customers[col].dropna().astype(str).str.contains(r'^\s+|\s+$', regex=True).sum() # Пошук аномальних пробілів на початку або в кінці рядка
    else:
        blank_cnt = 0
        edge_spaces_cnt = 0
        
    missing_stats_cust.append({
        'Колонка': col, 
        'Пропуски (NaN)': null_cnt, 
        'Частка NaN': round((null_cnt / len(customers)) * 100, 2),
        'Лише пробіли ("   ")': blank_cnt, 
        'Пробіли на краях': edge_spaces_cnt
    })

display(pd.DataFrame(missing_stats_cust))

print("\nДеталізація пропусків (NaN)")
for col in customers.columns:
    missing_mask = customers[col].isna()
    if missing_mask.sum() > 0:
        print(f"\nКолонка '{col}' має {missing_mask.sum()} пропущених значень. Ось ці рядки:")
        display(customers[missing_mask]) # Виводимо всі рядки з пропусками

# Виведення прикладів значень із зайвими пробілами
print("\nЗнайдені аномалії з пробілами")
for col in customers.columns:
    if ptypes.is_string_dtype(customers[col]) or customers[col].dtype == 'object':
        mask_spaces = customers[col].dropna().astype(str).str.contains(r'^\s+|\s+$', regex=True)
        if mask_spaces.sum() > 0:
            print(f"\nКолонка '{col}' має {mask_spaces.sum()} значень із зайвими пробілами:")
            display(customers[customers[col].notna()][mask_spaces][['customer_id', col]].head())

print("\nДУБЛІКАТИ (РЯДКИ ТА КЛЮЧІ)\n")
# Повні дублікати
full_dups_cust = customers[customers.duplicated(keep=False)]
print(f"Повні дублікати (всі стовпці збігаються): {len(full_dups_cust)}")
if not full_dups_cust.empty: display(full_dups_cust.sort_values(by=list(customers.columns)))

# Дублікати PK (customer_id)
key_dups_cust = customers[customers.duplicated(subset=['customer_id'], keep=False)]
print(f"Дублікати кандидатного ключа 'customer_id': {len(key_dups_cust)}")
if not key_dups_cust.empty: display(key_dups_cust.sort_values(by='customer_id'))

# Перевірка альтернативних ключів (email має бути унікальним)
email_dups = customers[customers.duplicated(subset=['email'], keep=False) & customers['email'].notna()]
print(f"Дублікати email адрес: {len(email_dups)}")
if not email_dups.empty: display(email_dups.sort_values(by='email'))

print("\nЧАСОВИЙ ДІАПАЗОН (registered_at)\n")
parsed_dates_cust = pd.to_datetime(customers['registered_at'], errors='coerce')
invalid_dates_cust = customers['registered_at'].notna() & parsed_dates_cust.isna()
print(f"Мінімальна дата реєстрації: {parsed_dates_cust.min()}")
print(f"Максимальна дата реєстрації: {parsed_dates_cust.max()}")
print(f"Пропущено дат (NaN): {customers['registered_at'].isna().sum()}")
print(f"Некоректні дати (не парсяться): {invalid_dates_cust.sum()}")
if invalid_dates_cust.sum() > 0: 
    print("Приклади некоректних дат:"); display(customers[invalid_dates_cust][['customer_id', 'registered_at']])


БАЗОВИЙ ОГЛЯД СТРУКТУРИ (customers.csv)



,customer_id,full_name,email,phone,city,service_zone_id,registered_at,customer_status
0,S06-C000001,Roman Moroz,s06.customer000001@example.test,3.809689e+11,Kyiv,KYI-Z04,2024-11-26T16:13:18.859859510Z,active
1,S06-C000002,Natalia Kovalchuk,s06.customer000002@example.test,3.809940e+11,Lviv,LVI-Z01,2025-05-23T23:57:47.626627069Z,active
2,S06-C000003,Viktor Oliinyk,s06.customer000003@example.test,3.809331e+11,Kharkiv,KHA-Z03,2024-10-20T16:43:24.424680013Z,active
3,S06-C000004,Volodymyr Kravets,s06.customer000004@example.test,3.809651e+11,Lviv,LVI-Z02,2024-08-21T20:27:12.333817581Z,active
4,S06-C000005,Pavlo Melnyk,s06.customer000005@example.test,3.809889e+11,Odesa,ODE-Z02,2025-11-12T23:36:49.574226944Z,inactive


,customer_id,full_name,email,phone,city,service_zone_id,registered_at,customer_status
796,S06-C000797,Viktoriia Oliinyk,s06.customer000797@example.test,3.806739e+11,Lviv,LVI-Z04,2025-12-11T22:08:52.206480554Z,active
797,S06-C000798,Viktoriia Oliinyk,s06.customer000798@example.test,3.809315e+11,Lviv,LVI-Z03,2026-07-26T17:36:32.695604156Z,active
798,S06-C000799,Ivan Hnatiuk,s06.customer000799@example.test,3.809850e+11,Odesa,ODE-Z01,2024-09-18T18:32:22.343647973Z,active
799,S06-C000800,Maksym Marchenko,s06.customer000800@example.test,3.806370e+11,Odesa,ODE-Z03,2025-03-29T06:27:45.608102746Z,active
800,S06-C000360,Artem Kravets,s06.customer000360@example.test,3.809362e+11,Lviv,LVI-Z01,2026-03-13T13:59:17.824468631Z,active


,customer_id,full_name,email,phone,city,service_zone_id,registered_at,customer_status
70,S06-C000071,Ivan Marchenko,s06.customer000071@example.test,3.809686e+11,Lviv,LVI-Z02,2025-09-06T18:08:59.819377271Z,active
507,S06-C000508,Oleksandr Lysenko,s06.customer000508@example.test,3.807316e+11,Dnipro,DNI-Z01,2026-07-31T23:52:24.193678598Z,active
152,S06-C000153,Kateryna Zakharchenko,s06.customer000153@example.test,3.806835e+11,Dnipro,DNI-Z02,2026-05-13T07:57:20.688928650Z,active


Розмірність (shape): (801, 8)
Колонки (columns): ['customer_id', 'full_name', 'email', 'phone', 'city', 'service_zone_id', 'registered_at', 'customer_status']

Типи даних (dtypes):


customer_id            str
full_name              str
email                  str
phone              float64
city                   str
service_zone_id        str
registered_at          str
customer_status        str
dtype: object

... (15 outputs left out)

,customer_id,full_name,email,phone,city,service_zone_id,registered_at,customer_status
359,S06-C000360,Artem Kravets,s06.customer000360@example.test,NaN,Lviv,LVI-Z01,2026-03-13T13:59:17.824468631Z,active
800,S06-C000360,Artem Kravets,s06.customer000360@example.test,3.809362e+11,Lviv,LVI-Z01,2026-03-13T13:59:17.824468631Z,active


Дублікати email адрес: 4


,customer_id,full_name,email,phone,city,service_zone_id,registered_at,customer_status
269,S06-C000270,Tetiana Hnatiuk,s06.customer000270@example.test,3.805066e+11,Kyiv,KYI-Z04,2025-12-04T12:10:52.213885720Z,active
702,S06-C000703,Danylo Polishchuk,s06.customer000270@example.test,3.809389e+11,Kyiv,KYI-Z05,2025-03-23T04:21:06.313382390Z,active
359,S06-C000360,Artem Kravets,s06.customer000360@example.test,NaN,Lviv,LVI-Z01,2026-03-13T13:59:17.824468631Z,active
800,S06-C000360,Artem Kravets,s06.customer000360@example.test,3.809362e+11,Lviv,LVI-Z01,2026-03-13T13:59:17.824468631Z,active



ЧАСОВИЙ ДІАПАЗОН (registered_at)

Мінімальна дата реєстрації: 2024-03-12 03:58:43.367625878+00:00
Максимальна дата реєстрації: 2026-08-28 09:37:27.449335604+00:00
Пропущено дат (NaN): 0
Некоректні дати (не парсяться): 0


In [3]:
import pandas.api.types as ptypes
from pathlib import Path
import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

orders = pd.read_csv(root / "data" / "raw" / "orders.csv")
customers = pd.read_csv(root / "data" / "raw" / "customers.csv")
merchants = pd.read_csv(root / "data" / "raw" / "merchants.csv")

print("\nБАЗОВИЙ ОГЛЯД СТРУКТУРИ (merchants.csv)\n")
display(merchants.head())
display(merchants.tail())
display(merchants.sample(3))
print(f"Розмірність (shape): {merchants.shape}")
print(f"Колонки (columns): {list(merchants.columns)}")
print("\nТипи даних (dtypes):"); display(merchants.dtypes)
print("\nІнформація (info):"); merchants.info()

print("\nОПИСОВА СТАТИСТИКА ТА КАТЕГОРІЇ\n")
display(merchants.describe(include='all'))
print("\nКількість унікальних значень у кожній колонці:"); display(merchants.nunique())

# Виводимо розподіл для категоріальних колонок
target_cols = ['merchant_type', 'city', 'merchant_status']
for col in target_cols:
    if col in merchants.columns:
        print(f"\nРозподіл для '{col}':")
        display(merchants[col].value_counts(dropna=False))

print("\nПРОПУСКИ, ПОРОЖНІ РЯДКИ ТА АНОМАЛЬНІ ПРОБІЛИ\n")
missing_stats_merch = []

for col in merchants.columns:
    null_cnt = merchants[col].isna().sum()
    is_text = ptypes.is_string_dtype(merchants[col]) or merchants[col].dtype == 'object'
    
    if is_text:
        blank_cnt = merchants[col].dropna().astype(str).str.strip().eq('').sum()
        edge_spaces_cnt = merchants[col].dropna().astype(str).str.contains(r'^\s+|\s+$', regex=True).sum()
    else:
        blank_cnt = 0
        edge_spaces_cnt = 0
        
    missing_stats_merch.append({
        'Колонка': col, 
        'Пропуски (NaN)': null_cnt, 
        'Частка NaN (%)': round((null_cnt / len(merchants)) * 100, 2),
        'Лише пробіли ("   ")': blank_cnt, 
        'Пробіли на краях': edge_spaces_cnt
    })

display(pd.DataFrame(missing_stats_merch))

print("\nДеталізація пропусків (NaN)")
has_nans_merch = False
for col in merchants.columns:
    missing_mask = merchants[col].isna()
    if missing_mask.sum() > 0:
        has_nans_merch = True
        print(f"\nКолонка '{col}' має {missing_mask.sum()} пропущених значень. Ось ці рядки:")
        display(merchants[missing_mask])

if not has_nans_merch:
    print("Пропусків у цій таблиці не виявлено.")

print("\nЗнайдені аномалії з пробілами")
for col in merchants.columns:
    if ptypes.is_string_dtype(merchants[col]) or merchants[col].dtype == 'object':
        mask_spaces = merchants[col].astype(str).str.contains(r'^\s+|\s+$', regex=True, na=False)
        if mask_spaces.sum() > 0:
            print(f"\nКолонка '{col}' має {mask_spaces.sum()} значень із зайвими пробілами:")
            display(merchants[mask_spaces][['merchant_id', col]].head())

print("\nПеревірка на проблеми з регістром (Case Sensitivity)")
for col in ['merchant_type', 'city']: # Перевіряємо колонки, де очікується єдиний стандарт
    if col in merchants.columns:
        # Створюємо словник: {значення_в_нижньому_регістрі: [оригінальні_значення]}
        unique_vals = merchants[col].dropna().unique()
        lower_map = {}
        for val in unique_vals:
            lower_val = str(val).lower()
            if lower_val not in lower_map:
                lower_map[lower_val] = []
            lower_map[lower_val].append(val)
        
        # Якщо одному нижньому регістру відповідає більше одного оригінального значення - це дублікат
        issues_found = False
        for lower_val, orig_vals in lower_map.items():
            if len(orig_vals) > 1:
                issues_found = True
                print(f"У колонці '{col}' знайдено дублювання через регістр: {orig_vals}")
                # Виводимо рядки, де зустрічається нестандартний варіант (не повністю в нижньому регістрі)
                for val in orig_vals:
                    if val != lower_val:
                        display(merchants[merchants[col] == val][['merchant_id', col]])
        
        if not issues_found:
            print(f"У колонці '{col}' проблем із регістром не виявлено.")

print("\nДУБЛІКАТИ (РЯДКИ ТА КЛЮЧІ)\n")
full_dups_merch = merchants[merchants.duplicated(keep=False)]
print(f"Повні дублікати (всі стовпці збігаються): {len(full_dups_merch)}")
if not full_dups_merch.empty: display(full_dups_merch.sort_values(by=list(merchants.columns)))

key_dups_merch = merchants[merchants.duplicated(subset=['merchant_id'], keep=False)]
print(f"Дублікати кандидатного ключа 'merchant_id': {len(key_dups_merch)}")
if not key_dups_merch.empty: display(key_dups_merch.sort_values(by='merchant_id'))

print("\nЧАСОВИЙ ДІАПАЗОН (opened_at)\n")
if 'opened_at' in merchants.columns:
    parsed_dates_merch = pd.to_datetime(merchants['opened_at'], errors='coerce')
    invalid_dates_merch = merchants['opened_at'].notna() & parsed_dates_merch.isna()
    print(f"Мінімальна дата відкриття: {parsed_dates_merch.min()}")
    print(f"Максимальна дата відкриття: {parsed_dates_merch.max()}")
    print(f"Пропущено дат (NaN): {merchants['opened_at'].isna().sum()}")
    print(f"Некоректні дати (не парсяться): {invalid_dates_merch.sum()}")
    if invalid_dates_merch.sum() > 0: 
        print("Приклади некоректних дат:"); display(merchants[invalid_dates_merch][['merchant_id', 'opened_at']])
else:
    print("Колонку 'opened_at' не знайдено.")

print("\nЕКСТРЕМАЛЬНІ ЧИСЛОВІ ЗНАЧЕННЯ\n")
num_cols_merch = merchants.select_dtypes(include=['number']).columns
if len(num_cols_merch) > 0:
    for col in num_cols_merch:
        neg = (merchants[col] < 0).sum()
        zeros = (merchants[col] == 0).sum()
        print(f"[{col:<20}] Min: {merchants[col].min():>8.2f} | Max: {merchants[col].max():>9.2f} | Нулів: {zeros:>4} | Від'ємних: {neg:>2}")
else:
    print("Числових колонок для аналізу не знайдено.")


БАЗОВИЙ ОГЛЯД СТРУКТУРИ (merchants.csv)



,merchant_id,merchant_name,merchant_type,city,service_zone_id,opened_at,merchant_status
0,S06-M00001,Family Point 01,restaurant,Odesa,ODE-Z02,2024-05-17T10:01:55.569007497Z,active
1,S06-M00002,Fresh Place 02,restaurant,Lviv,LVI-Z02,2020-08-24T23:26:08.474721377Z,active
2,S06-M00003,Morning Bakery 03,bakery,Kyiv,KYI-Z05,2019-09-26T12:36:44.017857737Z,active
3,S06-M00004,Daily Express 04,convenience,Lviv,LVI-Z01,2022-01-23T20:07:17.709382665Z,active
4,S06-M00005,Craft Point 05,restaurant,Kharkiv,KHA-Z02,2020-09-25T02:51:01.583987448Z,active


,merchant_id,merchant_name,merchant_type,city,service_zone_id,opened_at,merchant_status
38,S06-M00039,Golden Table 39,restaurant,Kyiv,KYI-Z02,2022-05-27T13:27:51.554972259Z,active
39,S06-M00040,Happy Paws 40,pet_store,Kyiv,KYI-Z02,2023-09-03T12:35:17.537091956Z,active
40,S06-M00041,Craft Kitchen 41,restaurant,Lviv,LVI-Z02,2023-02-11T22:54:47.569735777Z,active
41,S06-M00042,Golden Oven 42,bakery,Lviv,LVI-Z04,2020-12-19T05:29:00.054274975Z,active
42,S06-M00025,Local Bakery 25,bakery,Odesa,ODE-Z02,2024-04-01T23:19:34.358441229Z,active


,merchant_id,merchant_name,merchant_type,city,service_zone_id,opened_at,merchant_status
6,S06-M00007,Daily Grocer 07,grocery,Kyiv,KYI-Z03,2020-01-19T15:30:38.791823083Z,active
7,S06-M00008,Fresh Place 08,restaurant,Lviv,LVI-Z02,2021-11-04T16:54:00.124176864Z,active
25,S06-M00026,Fresh Table 26,restaurant,Kharkiv,KHA-Z01,2024-11-18T05:37:36.747281317Z,active


Розмірність (shape): (43, 7)
Колонки (columns): ['merchant_id', 'merchant_name', 'merchant_type', 'city', 'service_zone_id', 'opened_at', 'merchant_status']

Типи даних (dtypes):


merchant_id        str
merchant_name      str
merchant_type      str
city               str
service_zone_id    str
opened_at          str
merchant_status    str
dtype: object

... (15 outputs left out)

,merchant_id,merchant_name,merchant_type,city,service_zone_id,opened_at,merchant_status
24,S06-M00025,Local Bakery 25,bakery,Odesa,ODE-Z02,2024-04-01T23:19:34.358441229Z,active
42,S06-M00025,Local Bakery 25,bakery,Odesa,ODE-Z02,2024-04-01T23:19:34.358441229Z,active


Дублікати кандидатного ключа 'merchant_id': 2


,merchant_id,merchant_name,merchant_type,city,service_zone_id,opened_at,merchant_status
24,S06-M00025,Local Bakery 25,bakery,Odesa,ODE-Z02,2024-04-01T23:19:34.358441229Z,active
42,S06-M00025,Local Bakery 25,bakery,Odesa,ODE-Z02,2024-04-01T23:19:34.358441229Z,active



ЧАСОВИЙ ДІАПАЗОН (opened_at)

Мінімальна дата відкриття: 2019-09-26 12:36:44.017857737+00:00
Максимальна дата відкриття: 2025-07-29 10:54:44.349268384+00:00
Пропущено дат (NaN): 0
Некоректні дати (не парсяться): 0

ЕКСТРЕМАЛЬНІ ЧИСЛОВІ ЗНАЧЕННЯ

Числових колонок для аналізу не знайдено.


In [4]:
from pathlib import Path
import pandas as pd

orders = pd.read_csv(root / "data" / "raw" / "orders.csv")
customers = pd.read_csv(root / "data" / "raw" / "customers.csv")
merchants = pd.read_csv(root / "data" / "raw" / "merchants.csv")
products = pd.read_csv(root / "data" / "raw" / "products.csv")

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent


print("\nБАЗОВИЙ ОГЛЯД СТРУКТУРИ (products.csv)\n")
display(products.head())
display(products.tail())
display(products.sample(3))
print(f"Розмірність (shape): {products.shape}")
print(f"Колонки (columns): {list(products.columns)}")
print("\nТипи даних (dtypes):"); display(products.dtypes)
print("\nІнформація (info):"); products.info()

print("\nОПИСОВА СТАТИСТИКА ТА КАТЕГОРІЇ\n")
display(products.describe(include='all'))
print("\nКількість унікальних значень у кожній колонці:"); display(products.nunique())

# Перевірка розподілу популярних категорій
for col in ['product_category', 'is_active']:
    if col in products.columns:
        print(f"\nРозподіл для '{col}':")
        display(products[col].value_counts(dropna=False))

print("\nПРОПУСКИ, ПОРОЖНІ РЯДКИ ТА АНОМАЛЬНІ ПРОБІЛИ\n")
missing_stats_prod = []

for col in products.columns:
    null_cnt = products[col].isna().sum()
    is_text = ptypes.is_string_dtype(products[col]) or products[col].dtype == 'object'
    
    if is_text:
        # Рядки, що складаються ЛИШЕ з пробілів або абсолютно порожні
        blank_cnt = products[col].dropna().astype(str).str.strip().eq('').sum()
        # Пошук аномальних пробілів на початку або в кінці рядка
        edge_spaces_cnt = products[col].dropna().astype(str).str.contains(r'^\s+|\s+$', regex=True).sum()
    else:
        blank_cnt = 0
        edge_spaces_cnt = 0
        
    missing_stats_prod.append({
        'Колонка': col, 
        'Пропуски (NaN)': null_cnt, 
        'Частка NaN (%)': round((null_cnt / len(products)) * 100, 2),
        'Лише пробіли ("   ")': blank_cnt, 
        'Пробіли на краях': edge_spaces_cnt
    })

display(pd.DataFrame(missing_stats_prod))

print("\nДеталізація пропусків (NaN)")
has_nans_prod = False
for col in products.columns:
    missing_mask = products[col].isna()
    if missing_mask.sum() > 0:
        has_nans_prod = True
        print(f"\nКолонка '{col}' має {missing_mask.sum()} пропущених значень. Ось ці рядки:")
        display(products[missing_mask])

if not has_nans_prod:
    print("Пропусків (NaN) у цій таблиці не виявлено.")

# Виведення прикладів значень із зайвими пробілами
print("\nЗнайдені аномалії з пробілами")
for col in products.columns:
    if ptypes.is_string_dtype(products[col]) or products[col].dtype == 'object':
        # Надійна маска пошуку з na=False
        mask_spaces = products[col].astype(str).str.contains(r'^\s+|\s+$', regex=True, na=False)
        if mask_spaces.sum() > 0:
            print(f"\nКолонка '{col}' має {mask_spaces.sum()} значень із зайвими пробілами:")
            display(products[mask_spaces][['product_id', col]].head())

print("\nДУБЛІКАТИ (РЯДКИ ТА КЛЮЧІ)\n")
full_dups_prod = products[products.duplicated(keep=False)]
print(f"Повні дублікати (всі стовпці збігаються): {len(full_dups_prod)}")
if not full_dups_prod.empty: display(full_dups_prod.sort_values(by=list(products.columns)))

key_dups_prod = products[products.duplicated(subset=['product_id'], keep=False)]
print(f"Дублікати кандидатного ключа 'product_id': {len(key_dups_prod)}")
if not key_dups_prod.empty: display(key_dups_prod.sort_values(by='product_id'))

print("\nЧАСОВИЙ ДІАПАЗОН (updated_at)\n")
if 'updated_at' in products.columns:
    parsed_dates_prod = pd.to_datetime(products['updated_at'], errors='coerce')
    invalid_dates_prod = products['updated_at'].notna() & parsed_dates_prod.isna()
    print(f"Мінімальна дата оновлення: {parsed_dates_prod.min()}")
    print(f"Максимальна дата оновлення: {parsed_dates_prod.max()}")
    print(f"Пропущено дат (NaN): {products['updated_at'].isna().sum()}")
    print(f"Некоректні дати (не парсяться): {invalid_dates_prod.sum()}")
    if invalid_dates_prod.sum() > 0: 
        print("Приклади некоректних дат:"); display(products[invalid_dates_prod][['product_id', 'updated_at']])

print("\nЕКСТРЕМАЛЬНІ ЧИСЛОВІ ЗНАЧЕННЯ (Ціни)\n")
num_cols_prod = products.select_dtypes(include=['number']).columns
if len(num_cols_prod) > 0:
    for col in num_cols_prod:
        neg_mask = products[col] < 0
        zero_mask = products[col] == 0
        
        neg_count = neg_mask.sum()
        zero_count = zero_mask.sum()
        
        print(f"[{col:<15}] Min: {products[col].min():>8.2f} | Max: {products[col].max():>9.2f} | Нулів: {zero_count:>4} | Від'ємних: {neg_count:>2}")
        
        # Виводимо самі рядки, якщо знайшли нулі або від'ємні ціни
        if zero_count > 0:
            print(f"\nЗнайдено товари з нульовою ціною ({col}):")
            display(products[zero_mask])
            
        if neg_count > 0:
            print(f"\nЗнайдено товари з від'ємною ціною ({col}):")
            display(products[neg_mask])

print("\nПЕРЕВІРКА ЗВ'ЯЗКІВ (FOREIGN KEYS)\n")
# Перевірка: чи всі merchant_id у товарах існують у довіднику merchants
if 'merchant_id' in products.columns and 'merchant_id' in merchants.columns:
    missing_merchants_in_prod = products[~products['merchant_id'].isin(merchants['merchant_id'])]
    print(f"Товари з відсутніми (невідомими) merchant_id: {len(missing_merchants_in_prod)}")
    if not missing_merchants_in_prod.empty: 
        display(missing_merchants_in_prod[['product_id', 'merchant_id']].head())


БАЗОВИЙ ОГЛЯД СТРУКТУРИ (products.csv)



,product_id,merchant_id,product_name,category,unit_price_uah,is_active,updated_at
0,S06-P000001,S06-M00001,Tiramisu,desserts,118.25,True,2026-06-22T16:36:35.287736380Z
1,S06-P000002,S06-M00001,Classic Burger,burgers,154.27,True,2025-08-04T09:47:15.141255439Z
2,S06-P000003,S06-M00001,Veggie Burger,burgers,193.97,True,2025-12-09T21:28:11.572067155Z
3,S06-P000004,S06-M00001,Greek Salad,salads,156.22,True,2026-03-10T14:56:26.428222449Z
4,S06-P000005,S06-M00001,Caesar Salad,salads,151.60,True,2025-12-25T20:50:03.250341758Z


,product_id,merchant_id,product_name,category,unit_price_uah,is_active,updated_at
499,S06-P000500,S06-M00042,Croissant,pastries,104.05,True,2026-06-28T23:35:34.314746967Z
500,S06-P000501,S06-M00042,Muffin,pastries,111.69,True,2026-02-14T23:35:06.687293309Z
501,S06-P000502,S06-M00042,Muffin,pastries,73.54,True,2026-07-07T21:20:25.150409002Z
502,S06-P000503,S06-M00042,Croissant,pastries,71.32,False,2025-08-12T18:04:43.931705127Z
503,S06-P000315,S06-M00027,Coffee Beans,beverages,61.80,True,2026-03-26T07:06:49.499201464Z


,product_id,merchant_id,product_name,category,unit_price_uah,is_active,updated_at
36,S06-P000037,S06-M00003,Honey Cake,cakes,183.39,True,2026-04-03T00:56:24.175977647Z
216,S06-P000217,S06-M00018,Cheese Burger,burgers,261.28,True,2026-04-20T16:53:12.500955611Z
316,S06-P000317,S06-M00027,Baguette,bakery,25.18,True,2026-06-12T09:13:40.472346007Z


Розмірність (shape): (504, 7)
Колонки (columns): ['product_id', 'merchant_id', 'product_name', 'category', 'unit_price_uah', 'is_active', 'updated_at']

Типи даних (dtypes):


product_id            str
merchant_id           str
product_name          str
category              str
unit_price_uah    float64
is_active            bool
updated_at            str
dtype: object

... (14 outputs left out)


ЧАСОВИЙ ДІАПАЗОН (updated_at)

Мінімальна дата оновлення: 2025-08-02 05:21:33.091520344+00:00
Максимальна дата оновлення: 2026-08-30 18:28:07.799665384+00:00
Пропущено дат (NaN): 0
Некоректні дати (не парсяться): 0

ЕКСТРЕМАЛЬНІ ЧИСЛОВІ ЗНАЧЕННЯ (Ціни)

[unit_price_uah ] Min:     0.00 | Max:    778.33 | Нулів:    1 | Від'ємних:  0

Знайдено товари з нульовою ціною (unit_price_uah):


,product_id,merchant_id,product_name,category,unit_price_uah,is_active,updated_at
323,S06-P000324,S06-M00028,Juice,beverages,0.0,True,2026-04-20T05:33:50.867082593Z



ПЕРЕВІРКА ЗВ'ЯЗКІВ (FOREIGN KEYS)

Товари з відсутніми (невідомими) merchant_id: 1


,product_id,merchant_id
364,S06-P000365,S06-M99999
